# DAY4 실습 02 — 비율 검정 : 한 집단 · 두 집단

In [ ]:
%pip install -q koreanize-matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
from scipy import stats

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
data_folder = "/content/drive/MyDrive/HD-AI/data"
csv_path = os.path.join(data_folder, "ai4i2020.csv")

df = pd.read_csv(csv_path)

data = df.copy()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 데이터와 질문

데이터 파일 : `ai4i2020.csv`
출처 : UCI Machine Learning Repository - [AI4I 2020 Predictive Maintenance Dataset](https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset)

 한 행은 **운전 기록 1건**이다.

| 열 | 뜻 |
|---|---|
| `Type` | 제품 유형 (L : 저품질, M : 중품질, H : 고품질) |
| `Machine failure` | 고장 여부 (1 : 고장, 0 : 정상) |

**질문** : 운전 기록의 고장률은 기준과 다른가? 제품 유형에 따라 고장률이 다른가?

**실습 전제** : 기준 고장률은 학습을 위한 가상 기준이다. 운전 기록끼리 서로 독립이라고 가정한다.

In [ ]:
# 데이터 확인 : 행·열 수, 고장 여부와 제품 유형별 건수
print("행·열:", data.shape)                               # shape : (행 수, 열 수)
print(data["Machine failure"].value_counts())            # value_counts : 값마다 몇 건인지 센다. 0 : 정상, 1 : 고장
print(data["Type"].value_counts())                       # 제품 유형(L · M · H)별 건수

행·열: (10000, 14)
Machine failure
0    9661
1     339
Name: count, dtype: int64
Type
L    6000
M    2997
H    1003
Name: count, dtype: int64


## 1. 한 집단의 비율 : 정확 이항검정

**비율**은 전체 중 어떤 결과(예 : 불량, 고장)가 나온 건수의 몫이다. 결과가 불량/정상처럼 둘 중 하나인 자료에서, 표본 비율이 기준 비율과 다른지 검정할 때 **정확 이항검정**을 쓴다. 이항분포로 p값을 직접 계산하므로 정규성 확인이 필요 없다.

| 기호 | 뜻 |
|---|---|
| n | 시행 수 (검사한 개수) |
| k | 관심 결과(예 : 불량)가 나온 개수 |
| p̂ = k ÷ n | 표본 비율 |
| p₀ | 기준 비율 (귀무가설의 값) |

모비율의 기호 p는 p값과 글자가 같다. 가설 안의 p는 모비율, 결론의 p는 p값이다.

**가설 형식** (예 : 기준 불량률 5%)

- 귀무가설(H₀): 모집단 불량률은 5%다. (p = 0.05)
- 대립가설(H₁): 모집단 불량률은 5%가 아니다. (p ≠ 0.05)

**p값의 뜻** : 기준 불량률 5%가 맞다고 할 때, 지금 표본처럼 또는 지금보다 더 기준에서 벗어난 불량 수가 나올 확률이다. 이 확률이 0.05보다 작으면 "기준이 맞다고 보기 어렵다"고 판단한다.

**적용 조건** : 결과가 둘 중 하나이고, 시행 수 n이 정해져 있다
| 함수 | 기능 |
|---|---|
| `stats.binomtest(k=불량 수, n=전체 수, p=기준 비율)` | 정확 이항검정. `.pvalue` 로 p값을 꺼낸다 |
| `결과.proportion_ci(confidence_level=0.95)` | 모비율의 95% 신뢰구간. `.low`(아래 끝) · `.high`(위 끝) |

공식 문서 : [scipy.stats.binomtest](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.binomtest.html)

In [ ]:
# [예제] 이번 로트 200개 중 불량 16개다. 불량률이 기준 5%와 다른지 검정한다
# 귀무가설(H₀): 이번 로트가 속한 모집단의 불량률은 5%다. (p = 0.05)
# 대립가설(H₁): 이번 로트가 속한 모집단의 불량률은 5%가 아니다. (p ≠ 0.05)
# 유의수준 α = 0.05, 양측
k = 16          # 관심 결과(불량)가 나온 개수
n = 200         # 검사한 전체 수
p0 = 0.05       # 기준 불량률 (귀무가설의 값)

# 1) 표본 불량률 = 불량 수 ÷ 전체 수
print("표본 불량률:", round(k / n * 100, 1), "%")    # 비율(0 ~ 1)에 100을 곱해 % 로 바꾸고, round(값, 1) 로 소수 첫째 자리까지 반올림한다

# 2) 정확 이항검정
lot_result = stats.binomtest(k=k, n=n, p=p0)
#   binomtest(k=관심 결과 수, n=전체 수, p=기준 비율)
#   k 와 n 은 '개수'라 정수만 받는다. 16.0 처럼 소수점이 붙은 실수를 넣으면 오류가 난다 (k must be an integer)
#   p 는 비율이라 0 ~ 1 사이 실수로 넣는다. 5% → 0.05
#   alternative 를 적지 않으면 양측(기준과 다른가)으로 검정한다
#   lot_result 에는 검정 결과가 묶여 들어 있다. 그중 p값은 .pvalue 로 꺼낸다
print("p값:", round(lot_result.pvalue, 4))

# 3) 모비율의 95% 신뢰구간 : 모집단 불량률이 있을 만한 범위
lot_ci = lot_result.proportion_ci(confidence_level=0.95)   # confidence_level=0.95 → 95% 신뢰구간
#   .low 는 아래 끝, .high 는 위 끝이다. 비율(0 ~ 1)로 나오므로 100을 곱해 % 로 바꾼다
print("모비율 95% 신뢰구간:", round(lot_ci.low * 100, 1), "~", round(lot_ci.high * 100, 1), "%")

# [결과]
# - 표본 불량률 8.0% (기준보다 3.0%p 높다), p = 0.0708, 95% 신뢰구간 4.6 ~ 12.7%
# - 신뢰구간 4.6 ~ 12.7% 안에 기준 5% 가 들어 있다 → p값과 같은 판단이다
# - 통계적 결론 : p = 0.0708 ≥ 0.05 이므로 귀무가설을 기각하지 못한다
# - 사실적 결론 : 이번 로트의 불량률이 기준 5%와 다르다고 볼 통계적 근거가 부족하다 (표본 불량률 8.0%, n = 200)

표본 불량률: 8.0 %
p값: 0.0708
모비율 95% 신뢰구간: 4.6 ~ 12.7 %


In [ ]:
# [실습]
# 상황 : 설비팀은 운전 기록의 고장률 기준을 2%로 보고 있다. 운전 기록 500건을 뽑아 고장률이 기준과 다른지 검정한다.
#        귀무가설(H₀): 운전 기록의 모집단 고장률은 2%다. (p = 0.02)
#        대립가설(H₁): 운전 기록의 모집단 고장률은 2%가 아니다. (p ≠ 0.02)
#        유의수준 α = 0.05, 양측
# data 에서 500건(random_state=0)을 뽑아 고장 건수와 표본 고장률을 구하고, 정확 이항검정의 p값과 모비율의 95% 신뢰구간을 구하시오.



## 2. 독립된 두 집단의 비율 : 2×2 카이제곱 검정

두 집단(예 : 제품 유형 L과 H)의 비율이 같은지는 **2×2 교차표**로 정리해 **카이제곱 검정**으로 비교한다. 2×2는 행 2개(집단 2개) × 열 2개(결과 2가지)라는 뜻이다.

여러 집단의 비율이 같은지 보는 검정을 **카이제곱 동질성 검정**이라고 한다. 2×2 카이제곱 검정은 그중 집단이 두 개인 가장 단순한 경우다.

**가설 형식**

- 귀무가설(H₀): 두 집단의 모비율은 같다. (p₁ = p₂)
- 대립가설(H₁): 두 집단의 모비율은 다르다. (p₁ ≠ p₂)

**진행 순서**

| 단계 | 할 일 | 함수 |
|---|---|---|
| 2-1 교차표 | 집단별 정상 · 고장 건수를 세고 고장률을 구한다 | `pd.crosstab` |
| 2-2 조건 확인 | 기대빈도가 모든 칸에서 5 이상인지 본다 | `stats.contingency.expected_freq` |
| 2-3 검정 | χ²와 p값으로 두 비율이 같은지 판단한다 | `stats.chi2_contingency` |
| 2-4 차이의 크기 | 두 비율 차이의 95% 신뢰구간을 구한다 | `confint_proportions_2indep` |

공식 문서 : [pandas.crosstab](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html) · [scipy.stats.contingency.expected_freq](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.contingency.expected_freq.html) · [scipy.stats.chi2_contingency](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2_contingency.html) · [statsmodels confint_proportions_2indep](https://www.statsmodels.org/stable/generated/statsmodels.stats.proportion.confint_proportions_2indep.html)

### 2-1. 교차표 만들기

교차표는 두 범주형 열의 조합마다 건수를 센 표다. 여기서는 행이 제품 유형(L, H), 열이 고장 여부(0 = 정상, 1 = 고장)다. 교차표의 각 칸에 들어 있는 실제로 센 건수를 **관측빈도**라고 한다.

**교차표 읽는 법**

```
Machine failure     0    1
Type
L                5765  235
H                 982   21
```

| 부분 | 뜻 |
|---|---|
| `Type` | 행 이름 = 제품 유형 |
| `Machine failure` | 열 이름 = 고장 여부. 0 = 정상, 1 = 고장 |
| L 행 5765 · 235 | L 유형 기록 중 정상 5,765건, 고장 235건 |
| H 행 982 · 21 | H 유형 기록 중 정상 982건, 고장 21건 |

**고장률** = 고장 수 ÷ 전체 수. L은 235 ÷ 6,000 = 3.92%, H는 21 ÷ 1,003 = 2.09%다.

**%p(퍼센트포인트)** : 두 백분율의 차이를 나타내는 단위다. L 3.92%와 H 2.09%의 차이는 약 1.82%p다(반올림 전 값으로 계산). "1.82% 높다"라고 쓰면 L이 H의 1.82%만큼 높다는 뜻으로 오해할 수 있어 %p로 쓴다.

In [ ]:
# [예제] 2-1 제품 유형 L 과 H 의 교차표를 만들고 고장률을 구한다
#   data["Type"].isin(["L", "H"]) : 행마다 Type 이 L 또는 H 이면 True, 아니면(M) False
#   data.loc[True/False 목록] : True 인 행만 남긴다 → M 유형이 빠진다
lh = data.loc[data["Type"].isin(["L", "H"])]

#   pd.crosstab(행, 열) : 행 = 제품 유형, 열 = 고장 여부 (0 = 정상, 1 = 고장)
#   .loc[["L", "H"]] : 행 순서를 L, H 로 맞춘다
lh_table = pd.crosstab(lh["Type"], lh["Machine failure"]).loc[["L", "H"]]

print(lh_table)

l_fail, l_n = lh_table.loc["L", 1], lh_table.loc["L"].sum()   # L : 고장 수(1 칸), 전체 수(행 합계)
h_fail, h_n = lh_table.loc["H", 1], lh_table.loc["H"].sum()   # H : 고장 수(1 칸), 전체 수(행 합계)
#   a, b = 값1, 값2 : 오른쪽 두 값을 왼쪽 두 변수에 차례로 담는다
#   .loc["L", 1] 의 1 은 '두 번째 열'이 아니라 열 이름 1(고장)이다
print("L 고장 수:", l_fail, "/ 전체:", l_n, "/ 고장률:", round(l_fail / l_n * 100, 2), "%")
print("H 고장 수:", h_fail, "/ 전체:", h_n, "/ 고장률:", round(h_fail / h_n * 100, 2), "%")
print("고장률 차이(L − H):", round((l_fail / l_n - h_fail / h_n) * 100, 2), "%p")   # 두 비율을 뺀 뒤 100을 곱해 %p 로
f_diff = round((l_fail / l_n - h_fail / h_n) * 100, 2)
# [결과]
# - L 유형 고장률 3.92% (235 / 6,000), H 유형 고장률 2.09% (21 / 1,003)
# - 표본에서 L 이 H 보다 1.82%p 높다. 이 차이가 우연으로도 나올 만한 크기인지 2-2 ~ 2-4 단계로 확인한다

Machine failure     0    1
Type                      
L                5765  235
H                 982   21
L 고장 수: 235 / 전체: 6000 / 고장률: 3.92 %
H 고장 수: 21 / 전체: 1003 / 고장률: 2.09 %
고장률 차이(L − H): 1.82 %p


### 2-2. 조건 확인 : 기대빈도

**기대빈도**는 귀무가설(두 집단의 고장률이 같다)이 맞다면 각 칸에 나올 것으로 기대되는 건수다. 실제로 센 건수(관측빈도)와 비교하기 위한 기준이 된다.

**계산** : 칸마다 **행 합계 × 열 합계 ÷ 전체 합계**

| | 정상(0) | 고장(1) | 행 합계 |
|---|---|---|---|
| L | 5,765 | 235 | 6,000 |
| H | 982 | 21 | 1,003 |
| 열 합계 | 6,747 | 256 | 7,003 |

| 기대빈도 | 정상(0) | 고장(1) |
|---|---|---|
| L | 6,000 × 6,747 ÷ 7,003 = 5,780.7 | 6,000 × 256 ÷ 7,003 = 219.3 |
| H | 1,003 × 6,747 ÷ 7,003 = 966.3 | 1,003 × 256 ÷ 7,003 = 36.7 |

L · 고장 칸의 뜻 : 두 유형 모두 전체 고장률(256 ÷ 7,003 = 3.66%)과 같다면, L 6,000건 중 약 219.3건이 고장일 것이다. 실제로는 235건이다.

**조건** : 카이제곱 검정은 **기대빈도가 모든 칸에서 5 이상**일 때 p값을 믿을 수 있다. 관측빈도가 아니라 기대빈도를 본다. 5 미만인 칸이 있으면 Fisher 정확검정을 쓴다. 그래서 검정 전에 먼저 확인한다.

| 함수 | 기능 |
|---|---|
| `stats.contingency.expected_freq(교차표)` | 검정 없이 기대빈도만 계산한다 |


In [ ]:
# [예제] 2-2 기대빈도가 모든 칸에서 5 이상인지 확인한다
lh_expected = stats.contingency.expected_freq(lh_table)   # 기대빈도 = 행 합계 × 열 합계 ÷ 전체 합계
#   결과는 행 · 열 이름이 없는 숫자 배열이다.
#   lh_table.index 는 행 이름(L, H), lh_table.columns 는 열 이름(0, 1)이다. .round(1) : 소수 첫째 자리까지
print("관측빈도 (실제로 센 건수)")
print(lh_table)
print("기대빈도 (고장률이 같다면 기대되는 건수)")
print(pd.DataFrame(lh_expected, index=lh_table.index, columns=lh_table.columns).round(1))
print("가장 작은 기대빈도:", round(lh_expected.min(), 1))   # .min() : 네 칸 중 가장 작은 값. 이 값만 5 이상이면 모든 칸이 5 이상이다

# [결과]
# - 고장 칸 : L 은 관측 235 > 기대 219.3, H 는 관측 21 < 기대 36.7 → L 은 기대보다 고장이 많고 H 는 적다
# - 가장 작은 기대빈도 36.7 ≥ 5 → 조건을 만족하므로 카이제곱 검정을 쓴다

관측빈도 (실제로 센 건수)
Machine failure     0    1
Type                      
L                5765  235
H                 982   21
기대빈도 (고장률이 같다면 기대되는 건수)
Machine failure       0      1
Type                          
L                5780.7  219.3
H                 966.3   36.7
가장 작은 기대빈도: 36.7


### 2-3. 카이제곱 검정

**카이제곱 통계량** χ² = Σ (관측빈도 − 기대빈도)² ÷ 기대빈도

- 칸마다 "관측이 기대와 얼마나 어긋났는지"를 구해 모두 더한 값이다. Σ는 모두 더한다는 뜻이다.
- 관측과 기대가 똑같으면 0이다. 어긋날수록 커지고 p값은 작아진다.


| 함수 / 결과 | 기능 |
|---|---|
| `stats.chi2_contingency(교차표)` | 카이제곱 검정. 교차표를 그대로 넣는다. contingency table은 교차표를 뜻한다 |
| `결과.statistic` | χ² 값 |
| `결과.pvalue` | p값 |
| `결과.dof` | 자유도 (degrees of freedom) |
| `결과.expected_freq` | 기대빈도 표 (2-2에서 구한 값과 같다) |

In [ ]:
# [예제] 2-3 카이제곱 검정으로 L 과 H 의 고장률이 같은지 검정한다
# 귀무가설(H₀): L 유형과 H 유형의 모집단 고장률은 같다. (p_L = p_H)
# 대립가설(H₁): L 유형과 H 유형의 모집단 고장률은 다르다. (p_L ≠ p_H)
# 유의수준 α = 0.05, 양측
lh_result = stats.chi2_contingency(lh_table)
#   교차표를 그대로 넣는다. 고장률을 넣는 것이 아니라 건수 표를 넣는다
#   lh_result 에는 χ² · p값 · 자유도 · 기대빈도가 묶여 있다. 필요한 값을 . 뒤에 이름을 적어 꺼낸다
print("χ²:", round(lh_result.statistic, 2))         # statistic : 검정통계량 = χ²
print("자유도:", lh_result.dof)                     # (2 − 1) × (2 − 1) = 1
print("p값:", round(lh_result.pvalue, 4))

# [결과]
# - χ² = 7.60, 자유도 1, p = 0.0058
# - 통계적 결론 : p = 0.0058 < 0.05 이므로 귀무가설을 기각한다
# - 사실적 결론 : L 유형과 H 유형의 모집단 고장률이 같다고 보기 어렵다. 얼마나 다른지는 2-4 에서 신뢰구간으로 확인한다

χ²: 7.6
자유도: 1
p값: 0.0058


### 2-4. 차이의 크기 : 고장률 차이의 신뢰구간

p값은 "두 고장률이 다른가"에만 답한다. **얼마나 다른지**는 고장률 차이의 95% 신뢰구간으로 보고한다.

**계산** : `method="wald"`는 가장 기본 공식인 "고장률 차이 ± 1.96 × 표준오차"로 계산하라는 뜻이다. 표준오차는 표본 고장률이 표본마다 흔들리는 정도다.

**읽는 법**

| 구간 | 뜻 |
|---|---|
| 0을 포함하지 않는다 | 차이가 있다고 볼 수 있다 |
| 전체가 양수다 | 그룹1이 더 높다 |
| 전체가 음수다 | 그룹2가 더 높다 |
| 0을 포함한다 | 차이가 0일 수도 있다. 차이가 있다고 볼 근거가 부족하다 |

표본 차이(예 : 1.82%p)는 구간의 가운데에 있다. 그래서 "1.82 ± 1.01%p"처럼 ±로도 쓸 수 있다. 좌우 대칭인 Wald 구간에서만 해당한다.

| 함수 | 기능 |
|---|---|
| `confint_proportions_2indep(그룹1 고장 수, 그룹1 전체 수, 그룹2 고장 수, 그룹2 전체 수, method="wald")` | 독립된 두 집단의 비율 차이(그룹1 − 그룹2)의 95% 신뢰구간 (statsmodels). 아래 끝, 위 끝 두 값을 돌려준다 |

In [ ]:
# [예제] 2-4 고장률 차이(L − H)의 95% 신뢰구간을 구하고 결론을 쓴다
from statsmodels.stats.proportion import confint_proportions_2indep
#   from 모듈 import 함수 : statsmodels 라이브러리에서 이 함수만 불러온다 (scipy 에는 이 기능이 없다)

diff_low, diff_high = confint_proportions_2indep(l_fail, l_n, h_fail, h_n, method="wald")
#   (L 고장 수, L 전체 수, H 고장 수, H 전체 수) 순서로 넣으면 차이는 L − H 다. 순서를 바꾸면 부호가 반대가 된다
#   결과는 (아래 끝, 위 끝) 두 값이라 diff_low, diff_high 두 변수에 나눠 담는다
#   결과는 비율(0 ~ 1)이므로 100을 곱해 %p 로 바꾼다
print("고장률 L:", round(l_fail / l_n * 100, 2), "% / H:", round(h_fail / h_n * 100, 2), "%")
print("고장률 차이(L − H):", f_diff, "%p")
print("고장률 차이 95% 신뢰구간:", round(diff_low * 100, 2), "~", round(diff_high * 100, 2), "%p")
print("카이제곱 p값:", round(lh_result.pvalue, 4))

margin = (diff_high - diff_low) / 2          # 오차 범위 = 구간 폭(위 끝 − 아래 끝)의 절반 (Wald 처럼 좌우 대칭일 때만)
print("고장률 차이:",f_diff, "±", round(margin * 100, 2), "%p")

# [결과]
# - 고장률 차이 1.82 ± 1.01%p. Wald 구간은 좌우 대칭이라 구간 폭의 절반(1.01)을 ± 로 쓸 수 있다
# - 신뢰구간 0.81 ~ 2.84%p 는 0 을 포함하지 않고 전체가 양수다 → L 의 고장률이 더 높다
# - 통계적 결론 : p = 0.0058 < 0.05 이므로 귀무가설을 기각한다
# - 사실적 결론 : L 유형과 H 유형의 고장률은 다르다고 볼 통계적 근거가 있다. 표본에서 L 유형의 고장률(3.92%)이 H(2.09%)보다 1.82%p 높았다 (95% 신뢰구간 0.81 ~ 2.84%p)

고장률 L: 3.92 % / H: 2.09 %
고장률 차이(L − H): 1.82 %p
고장률 차이 95% 신뢰구간: 0.81 ~ 2.84 %p
카이제곱 p값: 0.0058
고장률 차이: 1.82 ± 1.01 %p


In [ ]:
# [실습]
# 상황 : 같은 순서로 제품 유형 M 과 H 의 고장률을 비교한다. 먼저 교차표를 만든다.
# M · H 유형의 교차표를 만들고(행 순서 M, H), 유형별 고장 수 · 전체 수 · 고장률과 고장률 차이(M − H)를 구하시오.



In [ ]:
# [실습]
# 상황 : 카이제곱 검정을 쓸 수 있는지 조건을 확인한다.
# M · H 교차표의 기대빈도 표를 행 · 열 이름이 붙은 표로 출력하고, 가장 작은 기대빈도를 구하시오.



In [ ]:
# [실습]
# 상황 : 조건을 만족하므로 검정하고 차이의 신뢰구간을 구한다.
#        귀무가설(H₀): M 유형과 H 유형의 모집단 고장률은 같다. (p_M = p_H)
#        대립가설(H₁): M 유형과 H 유형의 모집단 고장률은 다르다. (p_M ≠ p_H)
#        유의수준 α = 0.05, 양측
# 카이제곱 검정의 χ² · 자유도 · p값과 고장률 차이(M − H)의 95% 신뢰구간(method="wald")을 구하시오.



In [ ]:
# [실습]
# 상황 : 위 M · H 비교 결과를 보고서에 두 줄로 정리한다.
# 아래 결론 문장의 빈칸을 채우시오.
#    통계적 결론 : p = ______ ___ 0.05 이므로 귀무가설을 ___________.
#    사실적 결론 : M 유형과 H 유형의 고장률이 다르다고 볼 통계적 근거가 ______ (고장률 M ___% · H ___%, 차이 95% 신뢰구간 ___ ~ ___%p).



## 3. 참고 : 기대빈도가 5 미만일 때 — Fisher 정확검정

Fisher 정확검정은 표본 수가 적거나 기대빈도가 작은 경우, 범주형 변수 사이의 관련성을 확인하는 검정이다.

주로 2×2 분할표(교차표)에서 사용하며, 기대빈도가 5 미만인 칸이 있으면 카이제곱 검정 대신 고려할 수 있다.

| 함수 | 기능 |
|---|---|
| `stats.fisher_exact(교차표)` | 2×2 표의 정확 검정. `.pvalue` 로 p값을 꺼낸다 |
| `pd.concat([표1, 표2])` | 두 표를 위아래로 이어 붙인다 |

공식 문서 : [scipy.stats.fisher_exact](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.fisher_exact.html) · [pandas.concat](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)

In [ ]:
# [예제] 참고 : L · H 유형에서 각 60건만 뽑으면 고장 건수가 적어져 고장 칸의 기대빈도가 작아진다
# 귀무가설(H₀): L 유형과 H 유형의 모집단 고장률은 같다. (p_L = p_H)
# 대립가설(H₁): L 유형과 H 유형의 모집단 고장률은 다르다. (p_L ≠ p_H)
# 유의수준 α = 0.05, 양측
small = pd.concat([data.loc[data["Type"] == "L"].sample(n=60, random_state=3),
                   data.loc[data["Type"] == "H"].sample(n=60, random_state=3)])
#   data.loc[data["Type"] == "L"] : L 유형 행만 고른다.
#   .sample(n=60, random_state=3) : 그중 60건을 무작위로 뽑고, 실행할 때마다 같은 60건이 나오게 고정한다
#   pd.concat([표1, 표2]) : L 60건과 H 60건을 위아래로 이어 붙여 120건짜리 표를 만든다
small_table = pd.crosstab(small["Type"], small["Machine failure"]).loc[["L", "H"]]
print(small_table)

# 1) 조건 확인 : 기대빈도
small_expected = stats.contingency.expected_freq(small_table)
print(pd.DataFrame(small_expected, index=small_table.index, columns=small_table.columns).round(1))
print("가장 작은 기대빈도:", round(small_expected.min(), 1))   # 5 미만 → 카이제곱 대신 Fisher 정확검정

# 2) Fisher 정확검정
print("Fisher 정확검정 p값:", round(stats.fisher_exact(small_table).pvalue, 4))   # 2×2 건수 교차표를 그대로 넣고 .pvalue 로 p값을 꺼낸다

# [결과]
# - 고장 L 3건 · H 0건, 고장 칸 기대빈도 1.5 < 5 → Fisher 정확검정을 쓴다
# - p = 0.2437
# - 통계적 결론 : p = 0.2437 ≥ 0.05 이므로 귀무가설을 기각하지 못한다
# - 사실적 결론 : L 유형과 H 유형의 고장률이 다르다고 볼 통계적 근거가 부족하다 (각 60건)

Machine failure   0  1
Type                  
L                57  3
H                60  0
Machine failure     0    1
Type                      
L                58.5  1.5
H                58.5  1.5
가장 작은 기대빈도: 1.5
Fisher 정확검정 p값: 0.2437


## 4. 정리 : 비율 검정

| 질문                             | 검정          | 함수                            | 조건                                    |
| ------------------------------ | ----------- | ----------------------------- | ------------------------------------- |
| 한 집단의 비율이 기준과 다른가?             | 정확 이항검정     | `stats.binomtest(k=, n=, p=)` | 결과가 두 범주, 관측 독립, 성공 확률 동일             |
| 독립된 두 집단의 비율이 다른가?             | 2×2 카이제곱 검정 | `stats.chi2_contingency(교차표)` | 2×2 분할표, 관측 독립, 기대빈도 5 이상 권장          |
| 독립된 두 집단의 비율을 비교하는데 기대빈도가 작은가? | Fisher 정확검정 | `stats.fisher_exact(교차표)`     | 2×2 분할표, 관측 독립, 기대빈도 5 미만인 칸이 있을 때 고려 |

**두 집단 비율 비교 순서** : 교차표 → 기대빈도 확인(`stats.contingency.expected_freq`) → 검정 → 차이의 신뢰구간 → 두 줄 결론

### 해석 시 유의점

| 항목 | 해석 기준 |
|---|---|
| 기호 p | 가설 안의 p는 모비율, 결론의 p는 p값이다 |
| 비율 차이 | p값과 함께 표본 비율과 차이의 신뢰구간을 보고한다 |
| 기대빈도 | 교차표의 행 · 열 합계로 계산한다. 카이제곱을 쓸 수 있는지는 관측빈도가 아니라 기대빈도로 판단하며, 모든 칸이 5 이상인지 검정 전에 확인한다 |


공식 문서 : [scipy.stats.binomtest](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.binomtest.html) · [scipy.stats.contingency.expected_freq](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.contingency.expected_freq.html) · [scipy.stats.chi2_contingency](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2_contingency.html) · [scipy.stats.fisher_exact](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.fisher_exact.html)